# A2A MCP Client

Test client for the orchestrator agent. Run all agent notebooks first:
1. `mcp_server.ipynb` (port 10100)
2. `planner_agent.ipynb` (port 10102)
3. `travel_agents.ipynb` (ports 10103-10105)
4. `orchestrator_agent.ipynb` (port 10101)

In [1]:
import json
from uuid import uuid4

import httpx
from a2a.client import ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, SendMessageRequest, TaskState
from google.protobuf import json_format

ORCHESTRATOR_URL = 'http://localhost:10101'

## Fetch Agent Card

In [2]:
resp = httpx.get(f'{ORCHESTRATOR_URL}/.well-known/agent-card.json')
resp.raise_for_status()
print(json.dumps(resp.json(), indent=2))

{
  "name": "Orchestrator Agent",
  "description": "Orchestrates the task generation and execution",
  "supportedInterfaces": [
    {
      "url": "http://localhost:10101/",
      "protocolBinding": "JSONRPC"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": true
  },
  "defaultInputModes": [
    "text",
    "text/plain"
  ],
  "defaultOutputModes": [
    "text",
    "text/plain"
  ],
  "skills": [
    {
      "id": "executor",
      "name": "Task Executor",
      "description": "Orchestrates the task generation and execution, takes help from the planner to generate tasks",
      "tags": [
        "execute plan"
      ],
      "examples": [
        "Plan my trip to London, submit an expense report"
      ]
    }
  ],
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3",
  "url": "http://localhost:10101/"
}


## Run Conversation

In [3]:
async def run_conversation(initial_text: str, max_turns: int = 10):
    """Drive a multi-turn streaming conversation with the orchestrator."""
    task_id, context_id = None, None
    text = initial_text

    async with httpx.AsyncClient(timeout=httpx.Timeout(None)) as http:
        config = ClientConfig(httpx_client=http, streaming=True)
        client = await ClientFactory.connect(ORCHESTRATOR_URL, client_config=config)

        for turn in range(max_turns):
            print(f'\n===== turn {turn} =====')
            print(f'  >> {text}')

            message = Message(
                role=Role.ROLE_USER,
                parts=[Part(text=text)],
                message_id=uuid4().hex,
            )
            if task_id:
                message.task_id = task_id
            if context_id:
                message.context_id = context_id

            pending_question = None
            completed = False

            async for stream_resp, _task in client.send_message(SendMessageRequest(message=message)):
                payload_type = stream_resp.WhichOneof('payload')

                if payload_type == 'task':
                    t = stream_resp.task
                    task_id = t.id or task_id
                    context_id = t.context_id or context_id
                    print(f'[task] id={t.id} state={TaskState.Name(t.status.state)}')

                elif payload_type == 'status_update':
                    evt = stream_resp.status_update
                    task_id = evt.task_id or task_id
                    context_id = evt.context_id or context_id
                    msg_text = evt.status.message.parts[0].text if evt.status.message.parts else ''
                    state = TaskState.Name(evt.status.state)
                    print(f'[{state}] {msg_text}')
                    if evt.status.state == TaskState.TASK_STATE_INPUT_REQUIRED:
                        pending_question = msg_text
                    elif evt.status.state == TaskState.TASK_STATE_COMPLETED:
                        completed = True

                elif payload_type == 'artifact_update':
                    artifact = stream_resp.artifact_update.artifact
                    task_id = stream_resp.artifact_update.task_id or task_id
                    context_id = stream_resp.artifact_update.context_id or context_id
                    print(f'[artifact] {artifact.name}')
                    for part in artifact.parts:
                        content = part.WhichOneof('content')
                        if content == 'text':
                            print(part.text)
                        elif content == 'data':
                            print(json.dumps(json_format.MessageToDict(part.data), indent=2))

                elif payload_type == 'msg':
                    m = stream_resp.msg
                    task_id = m.task_id or task_id
                    context_id = m.context_id or context_id
                    print(f'[msg] {m.content[0].text if m.content else ""}')

            if completed:
                print('\n[conversation complete]')
                return
            if not pending_question:
                print('\n[stream ended without question or completion]')
                return

            print(f'\n>>> Agent asks: {pending_question}')
            text = input('Your answer: ').strip()
            if not text:
                print('[empty answer - stopping]')
                return

    print(f'\n[hit max_turns={max_turns}]')

In [4]:
await run_conversation('Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in a hotel suite and flying premium economy')


===== turn 0 =====
  >> Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in a hotel suite and flying premium economy
[task] id=7e497b1f-70d7-466c-9958-595e237b5908 state=TASK_STATE_SUBMITTED
[TASK_STATE_INPUT_REQUIRED] What is your total budget for this trip?

>>> Agent asks: What is your total budget for this trip?


Your answer:  30000



===== turn 1 =====
  >> 30000
[TASK_STATE_INPUT_REQUIRED] Is this trip for Business or Leisure?

>>> Agent asks: Is this trip for Business or Leisure?


Your answer:  leisure



===== turn 2 =====
  >> leisure
[TASK_STATE_INPUT_REQUIRED] How many travelers will be going on this trip?

>>> Agent asks: How many travelers will be going on this trip?


...


===== turn 4 =====
  >> keep
[TASK_STATE_INPUT_REQUIRED] Will you need a rental car during your stay in London?

>>> Agent asks: Will you need a rental car during your stay in London?


Your answer:  no



===== turn 5 =====
  >> no
[TASK_STATE_INPUT_REQUIRED] Since Premium Economy is not available, would you prefer BUSINESS class (premium service) or ECONOMY class (standard service) for your round-trip from SFO to LHR on May 1-11, 2026?

>>> Agent asks: Since Premium Economy is not available, would you prefer BUSINESS class (premium service) or ECONOMY class (standard service) for your round-trip from SFO to LHR on May 1-11, 2026?


Your answer:  economy



===== turn 6 =====
  >> economy
[TASK_STATE_WORKING] AirTicketingAgent: Processing...
[TASK_STATE_WORKING] AirTicketingAgent: Processing...
[TASK_STATE_WORKING] HotelBookingAgent: Processing...
[TASK_STATE_WORKING] HotelBookingAgent: Processing...
[artifact] Orchestrator Agent-result
## Travel Booking Summary

---

### Trip Overview
- **Trip Type:** Leisure
- **Travelers:** 2
- **Destination:** London, United Kingdom
- **Travel Dates:** May 1, 2026 – May 11, 2026 *(10 nights)*
- **Original Query:** *"Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in a hotel suite and flying premium economy"*

---

### ✈️ Flight Details

**Outbound Journey:**
| Detail | Information |
|---|---|
| **Route** | San Francisco (SFO) → London Heathrow (LHR) |
| **Date** | May 1, 2026 |
| **Airline & Flight** | Virgin Atlantic (VS) Flight 1731 |
| **Class** | Premium Economy |
| **Passengers** | 2 |
| **Cost** | $722.96 × 2 = **$1,445.92** |

**Return Journey:**
| Detail | Informa